In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import entropy
import numpy as np

In [6]:
df = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_LncRNA_identificados.xlsx')
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
df = df.loc[:, ~df.columns.str.contains('Raw')]
#trasponemos para que cada fila sea un paciente y cada columna sea una variable , incluimos el simbolo de hgnc como nombre de columnas  y eliminamos variables que son string
df = df.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T

##### Cálculo Información mutua y entropía

In [ ]:

entropias = {col: entropy(df[col]) for col in df.columns}

infoMutua = np.zeros((len(df.columns), len(df.columns)))

for i, col1 in enumerate(df.columns):
    for j, col2 in enumerate(df.columns):
        if i != j:  # No calcular para la misma variable
            infoMutua[i, j] = mutual_info_regression(df[[col1]], df[col2])[0]
infoMutua = pd.DataFrame(infoMutua, index=df.columns, columns=df.columns)

##### Cálculo correlaciones

In [20]:
correlacionesSpearman = df.corr(method= 'spearman') 
correlacionesSpearman.to_csv('../WorkingFiles/correlacionesSpearman.csv')

##### Comparación

In [23]:
infoMutua = pd.read_csv('../WorkingFiles/informacionMutuaVars.csv',index_col = 0)
correlacionesSpearman = pd.read_csv('../WorkingFiles/correlacionesSpearman.csv',index_col = 0)


In [ ]:
# Extraer pares únicos sin duplicados ni diagonal
corr_pairs = []

for i in range(len(df.columns)):
    for j in range(i+1, len(df.columns)):
        var1 = df.columns[i]
        var2 = df.columns[j]
        corr_val = abs(correlaciones_vars.iloc[i, j])
        corr_pairs.append(((var1, var2), corr_val))

# Ordenar por mayor correlación
ranking_corr = sorted(corr_pairs, key=lambda x: x[1], reverse=True)

In [45]:
# Extraer pares únicos sin duplicados
mi_pairs = []

for i in range(len(df.columns)):
    for j in range(i+1, len(df.columns)):
        var1 = df.columns[i]
        var2 = df.columns[j]
        # Puedes promediar MI(X→Y) y MI(Y→X)
        mi_val = (infoMutua.iloc[i, j] + infoMutua.iloc[j, i]) / 2
        mi_pairs.append(((var1, var2), mi_val))

# Ordenar por mayor información mutua
ranking_mi = sorted(mi_pairs, key=lambda x: x[1], reverse=True)

In [57]:
N = 100  # Número de pares a mostrar

# Convertimos ambas listas a DataFrames
df_corr = pd.DataFrame(ranking_corr[:N], columns=['pair', 'spearman_corr'])
df_mi = pd.DataFrame(ranking_mi[:N], columns=['pair', 'mutual_info'])

# Unimos por el par de variables
df_comparacion = pd.merge(df_corr, df_mi, on='pair', how='outer')

# Ordenamos por correlación (o si preferís, por mutual_info)
df_comparacion = df_comparacion.sort_values(by='spearman_corr', ascending=False).reset_index(drop=True)
df_comparacion.dropna()

,pair,spearman_corr,mutual_info
0,"(OLMALINC, PRKG1-AS1)",0.916807,0.945218
1,"(FGD5-AS1, NORAD)",0.890196,0.783128
2,"(OLMALINC, RTCA-AS1)",0.873389,0.623261
3,"(LINC02211, TTN-AS1)",0.857983,0.592570
4,"(LINC01151, LINC01844)",0.855462,0.566716
5,"(LINC02503, PRKG1-AS1)",0.854622,0.625931
6,"(PRKG1-AS1, RTCA-AS1)",0.848459,0.589087
7,"(RTCA-AS1, UGDH-AS1)",0.844258,0.674817
8,"(NORAD, RAP2C-AS1)",0.842297,0.541075
9,"(FTX, LINC00342)",0.840056,0.705670
